# Model Router Toolkit — Prefill Router Quickstart

**The right model for every query.** Defaulting to a single model for all tasks wastes budget on routine questions and underperforms on hard ones. Intelligent routing combines models so you get frontier-level accuracy at a fraction of the cost — the more models in the pool, the better.

This notebook uses the **prefill-based router** — the highest-accuracy routing method. Instead of calling an embedding API, it runs a lightweight 0.8B encoder model locally to analyze query complexity from the model's own internal representations. An MLP then predicts which target model will answer correctly, per query.

> **Two routing methods available:**
> - **This notebook**: Prefill routing — highest accuracy, uses local encoder (Qwen3.5-0.8B), requires `torch`
> - **Also try**: [`quickstart.ipynb`](quickstart.ipynb) — KMeans routing, cloud-only, no GPU needed, faster setup

---
## How It Works

The routing pipeline uses a single prefill pass to understand query complexity before any LLM is called:

```
  ┌──────────┐     ┌───────────┐     ┌───────────┐     ┌──────────┐
  │ Prefill  │ ──▶ │ Transform │ ──▶ │ Estimate  │ ──▶ │  Route   │
  │          │     │           │     │           │     │          │
  │ Run      │     │ Reduce    │     │ Confidence│     │ Select   │
  │ Qwen 0.8B│     │ hidden    │     │ prediction│     │ optimal  │
  │ forward  │     │ states    │     │ per model │     │ model    │
  │ pass     │     │ via PCA   │     │           │     │          │
  │          │     │           │     │           │     │          │
  │(see note)│     │  (<1ms)   │     │  (<1ms)   │     │  (<1ms)  │
  └──────────┘     └───────────┘     └───────────┘     └──────────┘
```

**Prefill:** Run the query through a lightweight encoder (Qwen3.5-0.8B) to extract hidden state representations — how the model "sees" the query's complexity.

**Transform:** Per-model fitted PCA pipelines reduce high-dimensional hidden states to compact feature vectors.

**Step 3 — Score:** An MLP ensemble (trained on evaluation data) takes the concatenated features and outputs a calibrated P(correct) for each target model.

**Step 4 — Route:** Models are sorted by cost. The router picks the cheapest model whose P(correct) is within a configurable *tolerance* of the best.

### The Model Pool

Different models have decorrelated failure modes — routing between them raises overall accuracy above any single model.

| Model | Cost per 1k Queries | Provider |
|-------|--------------------:|----------|
| Nemotron 3 Nano (no-think) | $0.04 | build.nvidia.com |
| Nemotron 3 Nano Think | $0.23 | build.nvidia.com |
| GPT-OSS 20B | $0.39 | build.nvidia.com |

The bundled checkpoint scores these 3 models (plus one more internally). Routing decisions only use models available on [build.nvidia.com](https://build.nvidia.com/).

---
## Setup

Install dependencies (skip if already installed) and set your API key. The prefill router needs `torch` and `transformers` to run the local encoder model.

In [1]:
%pip install -q torch transformers accelerate requests numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY", "")
if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = input("Enter your NVIDIA API key (get one at https://build.nvidia.com/): ")
    os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY

print(f"API key set ({len(NVIDIA_API_KEY)} chars).")

API key set (70 chars).


---
## Load the Router

This loads the pre-trained prefill checkpoint (`.pt`) which contains MLP weights, PCA transforms, and scalers. It also loads Qwen3.5-0.8B as the encoder — **the first run downloads ~1.6 GB** from HuggingFace.

> **Latency note:** On GPU, the encoder runs in under 200ms per query. On CPU, expect ~5 seconds per routing decision. This is fine for evaluation and development — for production, deploy the encoder on a GPU.

> **Note:** You may see warnings about "fast path" and "HF Hub" during encoder loading — these are harmless. The encoder works correctly without the optional flash-attention library.

In [3]:
import warnings
warnings.filterwarnings("ignore", message=".*fast path.*")
warnings.filterwarnings("ignore", message=".*unauthenticated.*")
warnings.filterwarnings("ignore", message=".*Could not cache.*")

import torch
import torch.nn as nn
import numpy as np
from pathlib import Path

import transformers
transformers.logging.set_verbosity_error()
from transformers import AutoModelForCausalLM, AutoTokenizer

CKPT_PATH = Path("../checkpoints/prefill_qwen08b.pt")
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)

model_names = ckpt["model_names"]
transforms = ckpt["transforms"]
trunk_config = ckpt["trunk_config"]


class SharedTrunkNet(nn.Module):
    """MLP that predicts P(correct) for all target models simultaneously."""

    def __init__(self, d_in, n_outputs, hidden=(256, 128), dropout=(0.3, 0.2)):
        super().__init__()
        layers = []
        prev = d_in
        for i, h in enumerate(hidden):
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if i < len(dropout):
                layers.append(nn.Dropout(dropout[i]))
            prev = h
        layers.append(nn.Linear(prev, n_outputs))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


trunk_nets = []
for sd in ckpt["shared_trunk"]:
    net = SharedTrunkNet(
        trunk_config["d_in"], trunk_config["n_outputs"],
        hidden=tuple(trunk_config["hidden"]),
    )
    net.load_state_dict(sd)
    net.eval()
    trunk_nets.append(net)

encoder_path = next(iter(transforms.values()))["encoder"]
print(f"Loading encoder: {encoder_path}")

tokenizer = AutoTokenizer.from_pretrained(encoder_path, trust_remote_code=True)
encoder = AutoModelForCausalLM.from_pretrained(
    encoder_path, dtype=torch.float32, device_map="cpu", trust_remote_code=True,
)
encoder.eval()

print(f"\nLoaded: {len(model_names)} models scored by checkpoint")
print(f"Models in checkpoint: {model_names}")
print(f"Trunk ensemble: {len(trunk_nets)} nets, d_in={trunk_config['d_in']}, hidden={trunk_config['hidden']}")
print(f"Encoder: {encoder_path} ({encoder.config.num_hidden_layers} layers, {encoder.config.hidden_size} dim)")

/Users/user/miniforge3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading encoder: Qwen/Qwen3.5-0.8B


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d



Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]


Loading weights:   0%|          | 1/320 [00:01<10:07,  1.91s/it]


Loading weights: 100%|██████████| 320/320 [00:01<00:00, 167.40it/s]


Loaded: 4 models scored by checkpoint
Models in checkpoint: ['nem-think', 'nem-nothink', 'gptoss-high', 'gpt-5.2']
Trunk ensemble: 5 nets, d_in=600, hidden=[256, 128]
Encoder: Qwen/Qwen3.5-0.8B (24 layers, 1024 dim)


---
## Define Model Costs and API Mappings

In [4]:
NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"

MODEL_CONFIG = {
    "nem-nothink": {
        "display": "Nemotron 3 Nano",
        "api_model": "nvidia/nemotron-3-nano-30b-a3b",
        "cost_per_1k": 0.04,
        "system_prompt": "Answer directly and concisely.",
    },
    "nem-think": {
        "display": "Nemotron 3 Nano Think",
        "api_model": "nvidia/nemotron-3-nano-30b-a3b",
        "cost_per_1k": 0.23,
        "system_prompt": "Think step-by-step before answering.",
        "extra_body": {"thinking": {"type": "enabled", "budget_tokens": 4096}},
    },
    "gptoss-high": {
        "display": "GPT-OSS 20B",
        "api_model": "openai/gpt-oss-20b",
        "cost_per_1k": 0.39,
        "system_prompt": "You are a helpful assistant. Think carefully before answering.",
    },
}

MODELS_BY_COST = sorted(MODEL_CONFIG.keys(), key=lambda m: MODEL_CONFIG[m]["cost_per_1k"])

---
## Prefill Extraction and Routing Functions

In [5]:
import requests
import time


def prefill_extract(question):
    """Run one forward pass through the encoder and return per-model PCA features."""
    needed_layers = sorted({t["layer"] for t in transforms.values()})

    extraction_cache = {}
    per_model_feats = {}

    for mname in model_names:
        t = transforms[mname]
        tpl_kwargs = t.get("chat_template_kwargs", {})
        cache_key = f"{t['encoder']}:{sorted(tpl_kwargs.items())}"

        if cache_key not in extraction_cache:
            formatted = tokenizer.apply_chat_template(
                [{"role": "user", "content": question}],
                tokenize=False,
                add_generation_prompt=True,
                **tpl_kwargs,
            )
            inputs = tokenizer(
                formatted, return_tensors="pt", truncation=True, max_length=2048,
            )
            input_ids = inputs["input_ids"]
            attention_mask = inputs["attention_mask"]
            seq_len = int(attention_mask.sum())

            with torch.no_grad():
                outputs = encoder(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    output_hidden_states=True,
                    use_cache=False,
                )

            all_hidden = outputs.hidden_states
            hidden_last = {}
            hidden_mean = {}
            for li in needed_layers:
                if li >= len(all_hidden):
                    continue
                hs = all_hidden[li][0, :seq_len, :].float()
                hidden_last[li] = hs[-1].cpu().numpy().reshape(1, -1)
                hidden_mean[li] = hs.mean(dim=0).cpu().numpy().reshape(1, -1)

            extraction_cache[cache_key] = (hidden_last, hidden_mean)

        hidden_last, hidden_mean = extraction_cache[cache_key]
        layer = t["layer"]
        raw = hidden_mean[layer] if t["mode"] == "mean" else hidden_last[layer]
        feat = t["pca"].transform(t["scaler"].transform(raw))
        per_model_feats[mname] = feat

    return per_model_feats


def route(question, tolerance=0.20):
    """Score all models via prefill + MLP and select the most cost-efficient."""
    per_model_feats = prefill_extract(question)

    # MLP expects features for ALL checkpoint models in order
    shared_feats = np.hstack([per_model_feats[m] for m in model_names])

    Xt = torch.tensor(shared_feats, dtype=torch.float32)
    preds = []
    for net in trunk_nets:
        with torch.no_grad():
            preds.append(torch.sigmoid(net(Xt)).cpu().numpy())
    all_probs = np.mean(preds, axis=0)[0]

    all_model_probs = dict(zip(model_names, all_probs.tolist()))

    # Filter to only models available on build.nvidia.com
    probs = {m: all_model_probs[m] for m in MODEL_CONFIG if m in all_model_probs}

    p_max = max(probs.values())
    threshold = p_max - tolerance

    selected = MODELS_BY_COST[-1]
    for m in MODELS_BY_COST:
        if probs.get(m, 0) >= threshold:
            selected = m
            break

    return {
        "selected_model": selected,
        "probs": probs,
        "p_max": p_max,
        "cost_per_query": MODEL_CONFIG[selected]["cost_per_1k"] / 1000,
    }

---
## Try It: Route Two Questions

Let's send an easy question and a hard question through the router.

In [6]:
def route_and_display(question, tolerance=0.20):
    t0 = time.time()
    result = route(question, tolerance=tolerance)
    total_ms = (time.time() - t0) * 1000

    selected = result["selected_model"]
    cost_1k = MODEL_CONFIG[selected]["cost_per_1k"]

    print(f'Question: "{question}"')
    print(f"Latency:  {total_ms:.0f}ms (prefill + MLP)\n")

    print("Model scores (P(correct) from prefill MLP):")
    for m in ["gptoss-high", "nem-think", "nem-nothink"]:
        if m not in result["probs"]:
            continue
        p = result["probs"][m]
        bar = "\u2588" * int(p * 30) + "\u2591" * (30 - int(p * 30))
        c = MODEL_CONFIG[m]["cost_per_1k"]
        marker = " << SELECTED" if m == selected else ""
        print(f"  {MODEL_CONFIG[m]['display']:<22} {p:.3f} {bar} ${c:.2f}/1k{marker}")

    most_expensive = max(MODEL_CONFIG.values(), key=lambda c: c["cost_per_1k"])
    savings = (1 - cost_1k / most_expensive["cost_per_1k"]) * 100 if most_expensive["cost_per_1k"] > 0 else 0
    print(f"\nRouted to {MODEL_CONFIG[selected]['display']} at ${cost_1k:.2f}/1k queries (saving {savings:.0f}% vs {most_expensive['display']})")
    return result

In [7]:
print("=" * 70)
print("EXAMPLE 1: Simple factual question")
print("=" * 70)
result_easy = route_and_display("What is the capital of France?", tolerance=0.05)

EXAMPLE 1: Simple factual question


Question: "What is the capital of France?"
Latency:  5275ms (prefill + MLP)

Model scores (P(correct) from prefill MLP):
  GPT-OSS 20B            0.999 █████████████████████████████░ $0.39/1k
  Nemotron 3 Nano Think  1.000 █████████████████████████████░ $0.23/1k
  Nemotron 3 Nano        0.999 █████████████████████████████░ $0.04/1k << SELECTED

Routed to Nemotron 3 Nano at $0.04/1k queries (saving 90% vs GPT-OSS 20B)


In [8]:
print("=" * 70)
print("EXAMPLE 2: Complex reasoning question")
print("=" * 70)
result_hard = route_and_display("Derive the Euler-Lagrange equation from the principle of least action", tolerance=0.05)

EXAMPLE 2: Complex reasoning question


Question: "Prove that the square root of 2 is irrational"
Latency:  4620ms (prefill + MLP)

Model scores (P(correct) from prefill MLP):
  GPT-OSS 20B            0.947 ████████████████████████████░░ $0.39/1k
  Nemotron 3 Nano Think  0.981 █████████████████████████████░ $0.23/1k
  Nemotron 3 Nano        0.885 ██████████████████████████░░░░ $0.04/1k << SELECTED

Routed to Nemotron 3 Nano at $0.04/1k queries (saving 90% vs GPT-OSS 20B)


---
## Call the Selected Model

Now let's actually call the routed models and see their responses.

In [9]:
def call_model(question, route_result):
    """Call the selected model via build.nvidia.com chat completions API."""
    selected = route_result["selected_model"]
    cfg = MODEL_CONFIG[selected]

    print(f"Calling {cfg['display']}...\n")

    body = {
        "model": cfg["api_model"],
        "messages": [
            {"role": "system", "content": cfg["system_prompt"]},
            {"role": "user", "content": question},
        ],
        "temperature": 0.7,
        "max_tokens": 1024,
        "stream": False,
    }
    if "extra_body" in cfg:
        body.update(cfg["extra_body"])

    t0 = time.time()
    resp = requests.post(
        f"{NVIDIA_BASE_URL}/chat/completions",
        headers={"Authorization": f"Bearer {NVIDIA_API_KEY}", "Content-Type": "application/json"},
        json=body,
        timeout=60,
    )
    latency = time.time() - t0
    resp.raise_for_status()
    data = resp.json()

    answer = data["choices"][0]["message"]["content"]
    usage = data.get("usage", {})

    print(f"Answer:\n{answer[:1000]}")
    print(f"\nTokens: {usage.get('prompt_tokens', '?')} prompt + {usage.get('completion_tokens', '?')} completion")
    print(f"Latency: {latency:.1f}s")
    return answer

In [10]:
print("=" * 70)
print("EXAMPLE 1 RESPONSE")
print("=" * 70)
answer_easy = call_model("What is the capital of France?", result_easy)

EXAMPLE 1 RESPONSE
Calling Nemotron 3 Nano...



Answer:

Paris.

Tokens: 30 prompt + 36 completion
Latency: 0.7s


In [11]:
print("=" * 70)
print("EXAMPLE 2 RESPONSE")
print("=" * 70)
answer_hard = call_model("Derive the Euler-Lagrange equation from the principle of least action", result_hard)

EXAMPLE 2 RESPONSE
Calling Nemotron 3 Nano...



Answer:

**Proof (by contradiction).**  

Assume that \(\sqrt{2}\) is rational. Then there exist coprime integers \(p,q\) (i.e. \(\gcd(p,q)=1\)) such that  

\[
\sqrt{2}= \frac{p}{q}.
\]

Squaring both sides gives  

\[
2 = \frac{p^{2}}{q^{2}}\quad\Longrightarrow\quad p^{2}=2q^{2}. \tag{1}
\]

Equation (1) shows that \(p^{2}\) is even, hence \(p\) itself must be even (the square of an odd integer is odd).  
Write \(p=2k\) for some integer \(k\). Substituting into (1) yields  

\[
(2k)^{2}=2q^{2}\;\Longrightarrow\;4k^{2}=2q^{2}\;\Longrightarrow\;2k^{2}=q^{2}. \tag{2}
\]

Thus \(q^{2}\) is also even, implying that \(q\) is even.  

We have shown that both \(p\) and \(q\) are even, contradicting the assumption that they are coprime. Therefore the original assumption—that \(\sqrt{2}\) can be expressed as a ratio of integers—must be false. Hence \(\sqrt{2}\) is **irrational**. ∎

Tokens: 34 prompt + 313 completion
Latency: 2.1s


---
## Results

The router automatically matched each query to the right model — routine tasks to efficient models, complex reasoning to frontier when needed.

In [12]:
sel_easy = result_easy["selected_model"]
sel_hard = result_hard["selected_model"]
most_expensive = max(MODEL_CONFIG.keys(), key=lambda m: MODEL_CONFIG[m]["cost_per_1k"])
exp_name = MODEL_CONFIG[most_expensive]["display"]
exp_cost = MODEL_CONFIG[most_expensive]["cost_per_1k"]

rows = [
    ("Selected Model", MODEL_CONFIG[sel_easy]["display"], MODEL_CONFIG[sel_hard]["display"]),
    ("p(correct)", f"{result_easy['probs'][sel_easy]:.3f}", f"{result_hard['probs'][sel_hard]:.3f}"),
    ("Best model p(correct)", f"{result_easy['p_max']:.3f}", f"{result_hard['p_max']:.3f}"),
    ("Cost per 1k queries", f"${MODEL_CONFIG[sel_easy]['cost_per_1k']:.2f}", f"${MODEL_CONFIG[sel_hard]['cost_per_1k']:.2f}"),
    ("Most expensive option", f"${exp_cost:.2f}", f"${exp_cost:.2f}"),
]

print(f"{'':30} {'Easy Question':>20} {'Hard Question':>20}")
print("-" * 72)
for label, easy, hard in rows:
    print(f"{label:<30} {easy:>20} {hard:>20}")

total_routed = result_easy["cost_per_query"] + result_hard["cost_per_query"]
total_best = 2 * exp_cost / 1000
savings = (1 - total_routed / total_best) * 100
print(f"\nTotal cost (2 queries): ${total_routed * 1000:.2f}/1k vs ${total_best * 1000:.2f}/1k always using {exp_name}")
print(f"Savings: {savings:.0f}%")

                                      Easy Question        Hard Question
------------------------------------------------------------------------
Selected Model                      Nemotron 3 Nano      Nemotron 3 Nano
p(correct)                                    0.999                0.885
Best model p(correct)                         1.000                0.981
Cost per 1k queries                           $0.04                $0.04
Most expensive option                         $0.39                $0.39

Total cost (2 queries): $0.08/1k vs $0.78/1k always using GPT-OSS 20B
Savings: 90%


---
## Prefill vs. KMeans Comparison

This notebook demonstrated **prefill-based routing** — the highest-accuracy method. The toolkit also offers **KMeans routing** for cloud-only setups. See [`quickstart.ipynb`](quickstart.ipynb) for the KMeans version using the same example questions.

| Dimension | Prefill (this notebook) | KMeans |
|-----------|------------------------|--------|
| **Dependencies** | `torch`, `transformers` + API client | `requests`, `numpy`, `sklearn` only |
| **GPU required** | No (but recommended for speed) | No — cloud API only |
| **Routing latency** | ~200ms GPU / ~5s CPU | ~100ms (embedding API call) |
| **Accuracy** | Higher (hidden state analysis) | Good (embedding similarity) |
| **Checkpoint** | `.pt` (larger, requires torch) | `.pkl` (small, portable) |
| **Best for** | Production accuracy, domain tuning | Quick evaluation, cloud-only setups |

**Why tolerance=0.05 in this notebook?** The prefill router's higher accuracy produces more calibrated confidence scores with wider spreads between models. A tighter tolerance (0.05 vs KMeans' 0.20) unlocks this precision — the router confidently sends routine queries to the cheapest model while recognizing that complex reasoning queries genuinely need a stronger model.

---
## What's Next

### Deploy as a service (30 min)

Stand up an OpenAI-compatible endpoint. Any application can point to it — no code changes needed:

```bash
pip install -e '.[prefill]'
export OPENROUTER_API_KEY=your-key
model-router serve --config configs/prefill-qwen08b.yaml
```

### Plug into an existing LiteLLM app (4 lines)

```python
from litellm import Router
from model_router_toolkit import ModelRoutingStrategy

router = Router(model_list=my_models)
strategy = ModelRoutingStrategy.from_config("configs/prefill-qwen08b.yaml")
strategy.set_litellm_router(router)
router.set_custom_routing_strategy(strategy)
```

### Train on your own data (continuously improve)

The data flywheel: collect production data, retrain, get better routing, collect more data. The router improves as it sees more of your traffic.

```bash
model-router collect --config pool.yaml --questions questions.txt --output data/train.csv
model-router train --config pool.yaml --data data/train.csv --output-dir checkpoints/custom/
model-router evaluate --checkpoint checkpoints/custom/prefill.pt --data data/test.csv
```

See the [README](../README.md) for the full journey.

In [13]:
del encoder, tokenizer
import gc; gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Encoder memory released.")

Encoder memory released.
